# Polaris API -> OpenSearch: every operation, every reachable status


**What this asks.** v1 (`polaris_log_coverage.ipynb`) asks *"if something went wrong on this
endpoint tomorrow, would there be a record of it?"* and answers it for 40 of the 63 operations,
at whatever status those calls happened to return. This notebook widens both axes: **every
operation the vendored specs name, at every status this build can actually produce**, verified
against the **OpenSearch** side of the pipeline with the gates of `GUIDE-schema-v3-testing.md`.

Plan: [`PLAN-api-status-matrix.md`](PLAN-api-status-matrix.md). It is the read, and it carries
the corrections its own predictions earned.

## Before you run this

```bash
kubectl -n <os-ns> port-forward svc/<opensearch> 9200:9200     # or reach it directly
export OPENSEARCH_PASS=...                                     # never in a config file
```

`local` env only, and it **mutates**: a catalog, principals, roles, namespaces, a table and a
view are created and deleted. ~300 calls.

## The two things that make this different from v1

**The sink is OpenSearch**, not VictoriaLogs, and they are fed by *different* Fluent Bits
running *different* schema versions (measured 2026-09-10: DaemonSet v3 -> OpenSearch,
Deployment v2 -> VictoriaLogs). Nothing verified on one carries to the other.

**The run is phased on window boundaries.** Every gate in the guide is per-window, so phases E,
F and G each get a window to themselves -- `last_write_bytes` means "the last 2xx POST in this
window" and that is only unambiguous when there is one.

## 0 -- Preflight

Nothing below is diagnosed until this passes. A check that cannot run says so; none of them
print a tidy zero.

In [1]:
import pathlib
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from polaris_test_utils import *  # noqa: F401,F403
import polaris_test_utils as ptu
import api_status_matrix as mx
import log_coverage as lc
import os_report as osr

init_env("local")
require_not_prod("the api-status-matrix drive creates and deletes catalogs, principals, roles, "
                 "namespaces, a table and a view, and provokes 500s")

RUN = str(int(time.time()))
STARTED = datetime.now(timezone.utc)
PRE, FAIL = {}, []


def sh(*args, timeout=25):
    """kubectl, when it is there. Returns stdout or None -- never raises."""
    if not shutil.which(args[0]):
        return None
    try:
        r = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip() if r.returncode == 0 else None
    except Exception:
        return None


OS_URL = f"https://{ptu.OPENSEARCH_HOST}:{ptu.OPENSEARCH_PORT}"
OS = osr.OSReports(
    OS_URL,
    auth=(ptu.OPENSEARCH_USER, ptu.OPENSEARCH_PASS),
    report_index=ptu.CFG.get("opensearch_report_index", osr.DEFAULT_REPORT_INDEX),
    logs_index=ptu.CFG.get("opensearch_logs_index", osr.DEFAULT_LOGS_INDEX),
)

try:
    ROOT_TOK = root_token()
    PRE["polaris"] = f"OK   {ptu.POLARIS_URL}"
except Exception as exc:
    ROOT_TOK = None
    FAIL.append(f"Polaris: {type(exc).__name__}: {exc}")

ok, detail = OS.ping()
PRE["opensearch"] = f"{'OK  ' if ok else 'DOWN'} {OS_URL}  ({detail})"
if not ok:
    FAIL.append(f"OpenSearch unreachable at {OS_URL} -- port-forward it, and check OPENSEARCH_PASS")

# DO THE INDICES EXIST AT ALL? README.md describes the OpenSearch shipper as a
# DIFFERENT Fluent Bit writing k8s-logs-*, so polaris-report-* / polaris-logs-*
# existing is a fact about THIS deployment and not an assumption the guide gets
# to make on our behalf.
IDX = {}
if ok:
    for row in OS.indices("polaris-*"):
        IDX[row["index"]] = row.get("docs.count")
    PRE["indices"] = ", ".join(f"{k} ({v})" for k, v in sorted(IDX.items())) or "NONE"
    if not any(k.startswith("polaris-report") for k in IDX):
        FAIL.append("no polaris-report-* index: gates 0-6 have no source. Nothing below would "
                    "fail -- it would all report zero, which is worse.")
    if not any(k.startswith("polaris-logs") for k in IDX):
        FAIL.append("no polaris-logs-* index: the per-call half of the matrix has no source.")

for k, v in PRE.items():
    print(f"  {k:22} {v}")
if FAIL:
    raise RuntimeError("PREFLIGHT FAILED:\n  - " + "\n  - ".join(FAIL))
print(f"\n  run={RUN}  started={STARTED.isoformat()}")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
  polaris                OK   http://192.168.139.2:8181
  opensearch             OK   https://localhost:9200  (HTTP 200 opensearch 3.5.0)
  indices                polaris-logs-2026.09.09 (61290), polaris-logs-2026.09.10 (672), polaris-report-2026.09.09 (1461), polaris-report-2026.09.10 (237)

  run=1789007773  started=2026-09-10T02:36:13.113785+00:00


## 0a -- Gate 7 FIRST: the Lua unit test

The guide is explicit that this runs **before** any query, and the reason is worth restating: it
separates *"the filter is wrong"* from *"my query is wrong"* in one step, and it has already
caught an invalid Lua escape that would have failed the whole chunk at load.

It needs the `local-k8s` checkout and a `lua5.4`. Both are on the machine running Jupyter and
neither is a reason to skip silently -- if it cannot run, that is recorded as **UNPROVEN** and
carried into the findings.

⚠ The two documents disagree about how many assertions this harness has: `SCHEMA-report.md` says
16, `GUIDE-schema-v3-testing.md` says 46. Whatever it prints is recorded verbatim; the count is
not asserted on.

In [2]:
GATE7 = {"ran": False, "ok": None, "detail": "", "assertions": None, "source": None}

LUA_TEST = pathlib.Path.home() / "hynix" / "local-k8s" / "logging" / "scripts" / "test-schema-v3.lua"
LUA_BIN = shutil.which("lua5.4") or shutil.which("lua")
EXTRACTED = pathlib.Path("/tmp/polaris.lua")


# THE PREREQUISITE THE GUIDE STATES AND THE FIRST VERSION OF THIS CELL DROPPED.
# `test-schema-v3.lua` runs the DEPLOYED SCRIPT TEXT -- that is the whole point
# of it -- and it reads that text from /tmp/polaris.lua. Run without the
# extraction it tests whatever happens to be at that path, or nothing at all,
# and reports a failure that reads "the filter is broken" when it means "you
# did not give me a filter". That is exactly what happened on the first run.
#
# The guide extracts from `fluent-bit/values.yaml`. This takes it from the
# RUNNING ConfigMap instead, for the reason the guide gives one line later:
# reading the file proves nothing about what is running. The key is located by
# CONTENT rather than by name, because charts name it differently and a rename
# would otherwise hand the test an empty script.
def _deployed_lua():
    raw = sh("kubectl", "-n", "datahub-hynix", "get", "cm",
             "-l", "app.kubernetes.io/instance=benchmarks-fluent-bit", "-o", "json")
    if not raw:
        return None, "kubectl unavailable, or no ConfigMap for benchmarks-fluent-bit"
    try:
        doc = json.loads(raw)
    except ValueError as exc:
        return None, f"the ConfigMap JSON did not parse: {exc}"
    best = None
    for item in doc.get("items", []):
        for key, value in (item.get("data") or {}).items():
            if not isinstance(value, str):
                continue
            if "polaris_noise_filter" in value or "build_report" in value:
                if best is None or len(value) > len(best[1]):
                    best = (key, value)
    if best is None:
        return None, "no ConfigMap key contained polaris_noise_filter or build_report"
    return best, ""


_pair, _why = _deployed_lua()
if _pair:
    EXTRACTED.write_text(_pair[1], encoding="utf-8")
    GATE7["source"] = f"ConfigMap key {_pair[0]} ({len(_pair[1])} bytes) -> {EXTRACTED}"
    print(f"  extracted {GATE7['source']}")
else:
    GATE7["detail"] = f"could not extract the deployed Lua: {_why}"

if not LUA_BIN:
    GATE7["detail"] = "no lua5.4 on PATH (brew install lua) -- UNPROVEN, not passed"
elif not LUA_TEST.exists():
    GATE7["detail"] = f"{LUA_TEST} not found -- UNPROVEN, not passed"
elif not _pair:
    pass
else:
    r = subprocess.run([LUA_BIN, str(LUA_TEST)], capture_output=True, text=True,
                       cwd=str(LUA_TEST.parent.parent.parent), timeout=120)
    _out = (r.stdout or "")
    if r.stderr:
        _out = _out + "\n" + r.stderr
    GATE7.update(ran=True, ok=(r.returncode == 0), detail=_out.strip()[-2500:])
    import re as _re
    _hit = _re.search(r"(\d+)\s+assertion", GATE7["detail"])
    GATE7["assertions"] = int(_hit.group(1)) if _hit else None

print(f"  Gate 7  {'PASS' if GATE7['ok'] else ('FAIL' if GATE7['ran'] else 'UNPROVEN')}")
print(f"          assertions reported: {GATE7['assertions']}  "
      "(SCHEMA-report.md says 16, the guide says 46 -- recorded, not asserted)")
for line in GATE7["detail"].splitlines()[-20:]:
    print(f"          {line}")

if not GATE7["ran"]:
    print()
    print("  Gate 7 is UNPROVEN, not passed. The rest still runs, and the findings cell records")
    print("  that the filter's own unit test never executed -- so a gate failure below cannot be")
    print("  attributed to the query rather than to the filter.")
elif not GATE7["ok"]:
    raise RuntimeError(
        "Gate 7 FAILED against the DEPLOYED script text (" + str(GATE7["source"]) + "). "
        "Read the assertion output above before anything else: it separates 'the filter is "
        "wrong' from 'my query is wrong' in one step, and every gate below would otherwise be "
        "measuring a filter that is known broken. If the failure names a missing function or a "
        "nil global, check that the ConfigMap key above is the WHOLE script rather than one "
        "fragment of a multi-key chart.")


  extracted ConfigMap key polaris_access_log.lua (34136 bytes) -> /tmp/polaris.lua
  Gate 7  PASS
          assertions reported: None  (SCHEMA-report.md says 16, the guide says 46 -- recorded, not asserted)
            [PASS] denied grant keeps its role row                got=true     want=true
            [PASS]   and records the denial                       got=1        want=1
            [PASS]   did NOT fall to the error bucket             got=nil      want=nil
            [PASS] role_keys_forced counts it                     got=1        want=1
          == __other__ split into overflow vs lost attribution ==
            [PASS] unknown-resource 404 -> __errors__             got=1        want=1
            [PASS]   kind is 'error'                              got=error    want=error
            [PASS]   api_kind mixed                               got=mixed    want=mixed
            [PASS]   NOT in __other__ (cap overflow)              got=nil      want=nil
          == a future AP

## 0b -- The policy that is RUNNING, read from the cluster

`load_policy(FB_VALUES_PATH)` reads the **Deployment's** values file, which is schema v2. This
notebook measures the **DaemonSet**, which is v3. Handing the v2 oracle to a v3 run reproduces
2026-09-04 exactly: eight "unexpected fields" per row, every row correct, nothing naming the
cause.

So the policy is read out of the running ConfigMap, and **both** shippers' versions are printed.
The ConfigMap is not the process either -- a pod that predates it is running the old text -- so
the pod start times are printed beside them.

In [3]:
SHIPPERS = {
    "benchmarks-fluent-bit": {"role": "DaemonSet -> OpenSearch  (THIS notebook)"},
    "fb-polaris-shipper": {"role": "Deployment -> VictoriaLogs  (v1's notebook)"},
}

import re

for rel, info in SHIPPERS.items():
    cm = sh("kubectl", "-n", "datahub-hynix", "get", "cm",
            "-l", f"app.kubernetes.io/instance={rel}", "-o", "yaml") or ""
    body = re.sub(r"--\[\[.*?\]\]", "", cm, flags=re.S)
    body = "\n".join(re.sub(r"--.*$", "", ln) for ln in body.splitlines())
    for const in ("SCHEMA_VERSION", "WINDOW_SECONDS", "REPORT_MAX_ROLE_KEYS"):
        hit = re.search(rf"local\s+{const}\s*=\s*(\d+)", body)
        info[const] = int(hit.group(1)) if hit else None
    info["pods"] = sh("kubectl", "-n", "datahub-hynix", "get", "pods",
                      "-l", f"app.kubernetes.io/instance={rel}",
                      "-o", "jsonpath={range .items[*]}{.metadata.name}@{.status.startTime} {end}")
    info["cm"] = cm
    print(f"  {rel:24} {info['role']}")
    print(f"  {'':24} SCHEMA_VERSION={info['SCHEMA_VERSION']}  "
          f"WINDOW_SECONDS={info['WINDOW_SECONDS']}  ROLE_KEYS={info['REPORT_MAX_ROLE_KEYS']}")
    print(f"  {'':24} {info['pods'] or 'kubectl unavailable -- a mid-run restart will NOT be detectable'}")

TARGET = SHIPPERS["benchmarks-fluent-bit"]
WINDOW_SECONDS = TARGET["WINDOW_SECONDS"]
DEPLOYED_SCHEMA = TARGET["SCHEMA_VERSION"]

if not WINDOW_SECONDS or not DEPLOYED_SCHEMA:
    raise RuntimeError("could not read WINDOW_SECONDS / SCHEMA_VERSION out of the running "
                       "ConfigMap. Everything below indexes windows by that constant; a caller "
                       "that guesses crosses no boundary and gets an EMPTY result rather than "
                       "an error.")
if DEPLOYED_SCHEMA != osr.SCHEMA_VERSION:
    raise RuntimeError(
        f"THE ORACLE IS A SCHEMA VERSION AWAY FROM THE FILTER: the DaemonSet emits v"
        f"{DEPLOYED_SCHEMA}, os_report.SCHEMA_VERSION is {osr.SCHEMA_VERSION}. Every "
        "'unexpected field' below would be that, and not drift in the pipeline.")

OS.schema_version = DEPLOYED_SCHEMA
print(f"\n  measuring schema v{DEPLOYED_SCHEMA}, windows of {WINDOW_SECONDS}s")
if SHIPPERS["fb-polaris-shipper"]["SCHEMA_VERSION"] != DEPLOYED_SCHEMA:
    print("  NOTE the two shippers disagree. That is expected as of 2026-09-10 and is exactly "
          "why\n       the policy is read per-shipper rather than from FB_VALUES_PATH.")

  benchmarks-fluent-bit    DaemonSet -> OpenSearch  (THIS notebook)
                           SCHEMA_VERSION=3  WINDOW_SECONDS=30  ROLE_KEYS=100
                           benchmarks-fluent-bit-5gsr8@2026-09-10T00:55:24Z
  fb-polaris-shipper       Deployment -> VictoriaLogs  (v1's notebook)
                           SCHEMA_VERSION=2  WINDOW_SECONDS=30  ROLE_KEYS=None
                           fb-polaris-shipper-fluent-bit-55b7bf586d-5kt5l@2026-09-07T04:21:51Z

  measuring schema v3, windows of 30s
  NOTE the two shippers disagree. That is expected as of 2026-09-10 and is exactly why
       the policy is read per-shipper rather than from FB_VALUES_PATH.


## 0c -- Gate 0: is v3 actually running

**PASS** -- a v3 document comes back and its `window_start` is recent.
**FAILS IF** -- no hits: the ConfigMap changed but the pod never rolled, or the roll never
happened. Reading `schema_version` out of a values file proves nothing, which is why this asks
the index.

Then the two mapping checks that decide whether any *number* below means anything.

In [4]:
newest = OS.latest_summary()
if not newest:
    raise RuntimeError(
        f"GATE 0 FAILED: no schema_version={OS.schema_version} document in "
        f"{OS.report_index}. The ConfigMap carries v{DEPLOYED_SCHEMA} and the index does not, "
        "so the pod never rolled or the report output is not wired to OpenSearch.")
print(f"  Gate 0  PASS   newest v{OS.schema_version} row: seq={newest.get('report_seq')} "
      f"window={newest.get('window_start')} host={newest.get('hostname')}")

# THE CHECK THAT DECIDES WHETHER EVERY SUM BELOW IS A NUMBER OR A LIE.
# `type_int_key` is set in the filter, but the INDEX decides, and it decides from
# the first document it ever saw. A text-typed `requests` makes gates 1-5 wrong
# while every query still returns 200.
NUMERIC_OK, KEYWORD_OK, MAPPING_NOTES = None, None, []
live = sorted([k for k in IDX if k.startswith("polaris-report")])[-1:]
live = live[0] if live else OS.report_index
try:
    OS.assert_numeric(index=live)
    NUMERIC_OK = True
    print(f"  mapping PASS   requests / errors / response_bytes are numeric in {live}")
except osr.MappingFault as exc:
    NUMERIC_OK = False
    MAPPING_NOTES.append(str(exc))
    print(f"  mapping FAIL   {exc}")

try:
    OS.assert_keyword_ready(index=live)
    KEYWORD_OK = True
except osr.MappingFault as exc:
    KEYWORD_OK = False
    MAPPING_NOTES.append(str(exc))
    print(f"  keyword FAIL   {exc}")

# The guide's own trap, checked rather than remembered.
MIN_RECORD_TYPE = OS.mapping_of("min_record_time", live)
print(f"  min_record_time mapped as {MIN_RECORD_TYPE!r} in {live}"
      + ("   <-- text: no date maths in this index, permanently (review #2)"
         if MIN_RECORD_TYPE not in ("date", None) else ""))
if MIN_RECORD_TYPE not in ("date", None):
    MAPPING_NOTES.append(
        f"min_record_time is {MIN_RECORD_TYPE} in {live}; the invariant is still checkable "
        "client-side (RFC3339 parses), but no range query or date histogram works on it here. "
        "Remedy: an index template for polaris-report-* typing min/max_record_time as date.")

if NUMERIC_OK is False:
    raise RuntimeError("the report fields are not numeric in the live index. Every sum in "
                       "gates 1-5 would be meaningless; fix the index template first.")

  Gate 0  PASS   newest v3 row: seq=202 window=2026-09-10T02:35:30Z host=benchmarks-fluent-bit-5gsr8
  mapping PASS   requests / errors / response_bytes are numeric in polaris-report-2026.09.10
  keyword FAIL   no .keyword subfield for ['window_start'] in polaris-report-2026.09.10: exact-match term filters on these silently match nothing.
  min_record_time mapped as 'text' in polaris-report-2026.09.10   <-- text: no date maths in this index, permanently (review #2)


## 1 -- Identities

Four, and the matrix needs all four. One identity would make the per-principal margin
unfalsifiable: `sum(principal.requests) == access_seen - parse_errors` is satisfied identically
by a global counter when there is only one row to sum.

In [5]:
from polaris_rest import PolarisREST

adm_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)

RUNNER_NAME, RUNNER_ROLE = f"mx_{RUN}_runner", f"mx_{RUN}_runner_role"
DENIED_NAME, DENIED_ROLE = f"mx_{RUN}_denied", f"mx_{RUN}_denied_role"

TOKENS, IDENTITY_NOTES = {mx.ADMIN: ROOT_TOK}, []

_cid, _sec = lc.provision_run_principal(adm_pc, RUNNER_NAME, RUNNER_ROLE, service_admin=True)
_r = adm_pc.get_token(_cid, _sec, scope=f"PRINCIPAL_ROLE:{RUNNER_ROLE}")
if _r.status_code == 200:
    TOKENS[mx.RUNNER] = _r.json()["access_token"]
    IDENTITY_NOTES.append(f"runner  {RUNNER_NAME}")
else:
    TOKENS[mx.RUNNER] = ROOT_TOK
    IDENTITY_NOTES.append(f"runner  FELL BACK TO ROOT ({_r.status_code}) -- the per-principal "
                          "margin is VOID this run, not failed")

_cid, _sec = lc.provision_run_principal(adm_pc, DENIED_NAME, DENIED_ROLE, fixture=None,
                                        service_admin=False)
_r = adm_pc.get_token(_cid, _sec, scope=f"PRINCIPAL_ROLE:{DENIED_ROLE}")
if _r.status_code == 200:
    TOKENS[mx.DENIED] = _r.json()["access_token"]
    IDENTITY_NOTES.append(f"denied  {DENIED_NAME} (a role, and no grants at all)")
else:
    TOKENS[mx.DENIED] = mx.GARBAGE_TOKEN
    IDENTITY_NOTES.append(f"denied  NO TOKEN ({_r.status_code}) -- every 403 cell will report "
                          "401 instead and be MISSED, correctly")

for n in IDENTITY_NOTES:
    print(f"  {n}")
print("  nobody  a syntactically valid bearer no realm accepts")

  runner  mx_1789007773_runner
  denied  mx_1789007773_denied (a role, and no grants at all)
  nobody  a syntactically valid bearer no realm accepts


## 2 -- The fixture, and the binding every cell is shaped against

One catalog, namespace, table and view. The `binding` is what turns a spec template into a URL;
`bind_path` raises rather than leaving a `{param}` behind, because a literal brace in a request
path is a 404 that looks like a finding.

In [6]:
import api_surface as surf
from iceberg_rest import IcebergREST, build_create_table_payload, build_schema

adm_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)
run_adm_ic = adm_ic  # the doomed family is created as root; the grid deletes it as itself
SCHEMA = build_schema([(1, "id", "long", True), (2, "val", "string", False)])

FX = surf.ProbeFixture(prefix=f"apimatrix{RUN}")
surf.setup_fixture(FX, adm_pc, adm_ic, ptu.BUCKET, ptu.MINIO_ENDPOINT, SCHEMA,
                   build_create_table_payload)
print(f"  fixture  {FX.cat} / {FX.ns} / {FX.tbl} / {FX.view}")

BINDING = {
    "catalog": FX.cat, "namespace": FX.ns, "table": FX.tbl, "view": FX.view,
    "principalName": RUNNER_NAME, "principalRoleName": RUNNER_ROLE,
    "catalogRoleName": f"mx_{RUN}_crole", "plan-id": "no-such-plan-id",
    # names the grid CREATES rather than reads
    "new_catalog": f"mx{RUN}cat2", "new_namespace": f"mx_{RUN}_ns2",
    "new_table": f"mx_{RUN}_tbl2", "new_view": f"mx_{RUN}_vw2",
    "new_principal": f"mx_{RUN}_p2", "new_principal_role": f"mx_{RUN}_prole2",
    "new_catalog_role": f"mx_{RUN}_crole2",
    "renamed_table": f"{FX.tbl}_renamed", "renamed_view": f"{FX.view}_renamed",
    # THE DOOMED FAMILY. Every destructive operation in the grid is pointed
    # here by api_status_matrix.REBIND, and nothing else in the run reads them.
    "doomed_catalog": f"mx{RUN}doomed", "doomed_namespace": f"mx_{RUN}_ns_doomed",
    "doomed_table": f"mx_{RUN}_tbl_doomed", "doomed_view": f"mx_{RUN}_vw_doomed",
    "doomed_principal": f"mx_{RUN}_p_doomed",
    "doomed_principal_role": f"mx_{RUN}_prole_doomed",
    "doomed_catalog_role": f"mx_{RUN}_crole_doomed",
    "entity_version": 1, "run": RUN,
    "base_location": f"s3a://{ptu.BUCKET}/mx{RUN}cat2/",
    "allowed_location": f"s3a://{ptu.BUCKET}/",
    "s3_endpoint": ptu.MINIO_ENDPOINT, "s3_endpoint_internal": ptu.MINIO_ENDPOINT_INTERNAL,
    "metadata_location": f"s3a://{ptu.BUCKET}/mx{RUN}cat2/never-registered/metadata.json",
    "client_id": ptu.ROOT_CLIENT, "client_secret": ptu.ROOT_SECRET,
}

# The catalog role the role-graph cells READ has to exist before they read it.
_r = adm_pc.create_catalog_role(FX.cat, BINDING["catalogRoleName"])
print(f"  catalog role {BINDING['catalogRoleName']}: {_r.status_code}")

# THE DOOMED FAMILY. The happy sweep drives deleteCatalog, dropTable, dropView,
# dropNamespace, deletePrincipal and the two revokes -- bound naively they take
# the FIXTURE, and the run destroys itself at about call 30 of 63 with every
# later cell 404ing. api_status_matrix.REBIND points them here instead, so the
# deletes are real deletes of real entities and nothing the run still needs.
SETUP = {}
SETUP["doomed catalog"] = adm_pc.create_catalog(
    BINDING["doomed_catalog"], ptu.BUCKET, ptu.MINIO_ENDPOINT,
    minio_endpoint_internal=ptu.MINIO_ENDPOINT_INTERNAL,
).status_code
SETUP["doomed namespace"] = run_adm_ic.create_namespace(
    FX.cat, BINDING["doomed_namespace"]).status_code
SETUP["doomed table"] = run_adm_ic.create_table(
    FX.cat, FX.ns, build_create_table_payload(BINDING["doomed_table"], SCHEMA)).status_code
SETUP["doomed view"] = run_adm_ic.create_view(FX.cat, FX.ns, {
    "name": BINDING["doomed_view"], "schema": SCHEMA,
    "view-version": {"version-id": 1, "timestamp-ms": 0, "schema-id": 0,
                     "summary": {"engine-name": "api-status-matrix"},
                     "default-namespace": [FX.ns],
                     "representations": [{"type": "sql", "sql": "SELECT 1",
                                          "dialect": "spark"}]},
    "properties": {"mx.run": RUN}}).status_code
SETUP["doomed principal"] = adm_pc.create_principal(BINDING["doomed_principal"]).status_code
SETUP["doomed principal role"] = adm_pc.create_principal_role(
    BINDING["doomed_principal_role"]).status_code
SETUP["doomed catalog role"] = adm_pc.create_catalog_role(
    FX.cat, BINDING["doomed_catalog_role"]).status_code
# the two revokes need something assigned to revoke
SETUP["assign prole->principal"] = adm_pc.assign_principal_role_to_principal(
    BINDING["doomed_principal"], BINDING["doomed_principal_role"]).status_code
# ARGUMENT ORDER IS (catalog, principal_role, catalog_role) -- and this PUT can
# 404 immediately after create_catalog_role on PG-HA read-after-write lag, which
# is a known signature here, not a finding. Retry rather than record a false one.
SETUP["assign crole->prole"] = None
for _try in range(4):
    _ar = adm_pc.assign_catalog_role_to_principal_role(
        FX.cat, BINDING["doomed_principal_role"], BINDING["doomed_catalog_role"])
    SETUP["assign crole->prole"] = _ar.status_code
    if _ar.status_code != 404:
        break
    time.sleep(0.75)

for k, v in SETUP.items():
    print(f"  {k:26} {v}")
_bad = {k: v for k, v in SETUP.items() if not (200 <= int(v) < 300)}
if _bad:
    print(f"  !! setup did not fully succeed: {_bad}")
    print("     the delete cells for those entities will report 404 and be MISSED --")
    print("     correctly, and as a harness gap rather than a pipeline finding.")

  fixture  apimatrix1789007773_cat / probe_ns / probe_tbl / probe_view
  catalog role mx_1789007773_crole: 201
  doomed catalog             201
  doomed namespace           500
  doomed table               200
  doomed view                500
  doomed principal           201
  doomed principal role      201
  doomed catalog role        201
  assign prole->principal    201
  assign crole->prole        201
  !! setup did not fully succeed: {'doomed namespace': 500, 'doomed view': 500}
     the delete cells for those entities will report 404 and be MISSED --
     correctly, and as a harness gap rather than a pipeline finding.


## 3 -- The grid, printed BEFORE anything is driven

The denominator is a fact about the vendored specs and does not need this run to establish it.
Printing it first means a reader can see what was planned even if the drive falls over halfway.

The **not-reachable ledger** prints beside it. `settled` entries have a structural reason;
`probe` entries are assumptions that get one call each, recorded either way.

In [7]:
import collections

OPS = mx.load_spec(pathlib.Path.cwd() / "spec")
GRID = mx.cells(OPS)
BY_TARGET = collections.Counter(c.target for c in GRID)
BY_PHASE = collections.Counter(c.phase for c in GRID)

print(f"  {len(OPS)} operations from log-coverage/spec/  "
      f"({sum(1 for o in OPS if o.api == 'management')} management, "
      f"{sum(1 for o in OPS if o.api == 'catalog')} catalog)")
print(f"  {len(GRID)} cells   " + "  ".join(f"{k}={v}" for k, v in sorted(BY_TARGET.items(), key=str)))
print("  phases   " + "  ".join(f"{k}={v}" for k, v in sorted(BY_PHASE.items())))

print("\n  NOT REACHABLE on this build:")
for row in mx.ledger():
    mark = "settled" if row["state"] == "settled" else "PROBE  "
    print(f"    {str(row['status']):>4}  {mark}  {row['why']}")
print(f"  EXTRAS the spec never declares but this build returns: {mx.UNDECLARED_EXTRAS}")

# Every cell must build a request before ANY of them is issued. A missing payload
# or an unbindable path is a harness bug, and finding it at call 200 of 286 wastes
# the run's windows.
_dry = [mx.request_for(c, BINDING, TOKENS, "dry-run", ptu.REALM) for c in GRID]
print(f"\n  dry run  {len(_dry)} requests built, none raised")

ROWS, PHASE_WINDOWS = [], {}


def run_phase(name, cells_, note=""):
    """Drive one phase, alone in its window, and record the window it fell in."""
    wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
    print(f"  phase {name}: {len(cells_)} cells; waiting {wait:.0f}s for a fresh window {note}")
    time.sleep(wait)
    t0 = time.time()
    rows = mx.drive(cells_, BINDING, TOKENS, ptu.POLARIS_URL, ptu.REALM, RUN,
                    seq_from=2000 + len(ROWS))
    t1 = time.time()
    w0, _ = osr.window_bounds(t0, WINDOW_SECONDS)
    w1, _ = osr.window_bounds(t1, WINDOW_SECONDS)
    PHASE_WINDOWS[name] = {"start": w0, "end": w1, "straddled": w0 != w1,
                           "seconds": round(t1 - t0, 1), "cells": len(cells_)}
    ROWS.extend(rows)
    got = collections.Counter(r["verdict"] for r in rows)
    print(f"           {t1 - t0:.1f}s in window {w0}"
          + ("  !! STRADDLED -- this phase's window checks are VOID, not failed"
             if w0 != w1 else "")
          + f"   {dict(got)}")
    return rows

  63 operations from log-coverage/spec/  (33 management, 30 catalog)
  286 cells   2=63  400=28  401=63  403=62  404=55  409=15
  phases   B=63  C=180  D=43

  NOT REACHABLE on this build:
     304  PROBE    needs If-None-Match support on loadTable
     406  PROBE    needs content negotiation to refuse a JSON-only endpoint
     419  PROBE    Iceberg's credential-refresh code; this build is expected to answer 401 instead
     429  settled  rateLimiter.type is no-op on this build -- the limiter cannot produce a 429
     502  settled  nothing proxies Polaris here; a port-forward is not a gateway
     503  settled  the only route is scaling Polaris, and HPA movement invalidates the run (local-k8s #8)
     504  settled  same -- no gateway to time out
     5XX  settled  a spec placeholder, not a status
  EXTRAS the spec never declares but this build returns: (405, 500)

  dry run  286 requests built, none raised


## 4 -- Phase B: the 2xx sweep

Every operation at its own success code, adjudicated against what the spec **declares** --
`createCatalog` answers 201 and `dropNamespace` 204, so a matrix expecting 200 would call both
a miss.

In [8]:
PHASE_B = run_phase("B", [c for c in GRID if c.phase == "B" and c.target == 2])
_missed = [r for r in PHASE_B if r["verdict"] == mx.MISSED]
print(f"\n  {len(PHASE_B) - len(_missed)}/{len(PHASE_B)} covered")
for r in _missed[:20]:
    print(f"    MISS  {r['op_id']:34} got {r['status']}  {r['why']}")

  phase B: 63 cells; waiting 4s for a fresh window 
           1.0s in window 2026-09-10T02:36:30Z   {'covered': 32, 'missed': 31}

  32/63 covered
    MISS  listNamespaces                     got 403  expected one of (200,)
    MISS  createNamespace                    got 403  expected one of (200,)
    MISS  dropNamespace                      got 403  expected one of (204,)
    MISS  loadNamespaceMetadata              got 403  expected one of (200,)
    MISS  namespaceExists                    got 403  expected one of (204,)
    MISS  updateProperties                   got 403  expected one of (200,)
    MISS  registerTable                      got 403  expected one of (200,)
    MISS  listTables                         got 403  expected one of (200,)
    MISS  createTable                        got 403  expected one of (200,)
    MISS  dropTable                          got 403  expected one of (204,)
    MISS  loadTable                          got 403  expected one of (200,)
    M

## 5 -- Phase C: 401 / 403 / 404

The error sweep. Rule 3 keeps **every one of these** as an individual record, so this is also
the phase that proves the per-call half of the pipeline: every cell here must be findable in
`polaris-logs-*` by its request id.

Gate 5 reads this window: these errors land in `__errors__` (attribution lost, record kept),
and `__other__` must stay absent or zero -- no window here carries 500 distinct resources.

In [9]:
PHASE_C = run_phase("C", [c for c in GRID if c.phase == "C"])
_by = collections.Counter((r["target"], r["status"]) for r in PHASE_C)
print("\n  target -> actual:")
for (t, a), n in sorted(_by.items(), key=lambda kv: (str(kv[0][0]), str(kv[0][1]))):
    mark = "" if t == a else "   <-- MISS"
    print(f"    {t:>4} -> {str(a):>4}  x{n}{mark}")

  phase C: 180 cells; waiting 3s for a fresh window 
           1.2s in window 2026-09-10T02:37:00Z   {'covered': 158, 'missed': 22}

  target -> actual:
     401 ->  401  x59
     401 ->  404  x4   <-- MISS
     403 ->  200  x1   <-- MISS
     403 ->  204  x1   <-- MISS
     403 ->  403  x45
     403 ->  404  x15   <-- MISS
     404 ->  204  x1   <-- MISS
     404 ->  404  x54


## 6 -- Phase D: 409 / 400

409 comes from **order** on the creates -- phase B already made these -- and from a stale
`currentEntityVersion` on the management PUTs. 400 comes from a body missing its required
fields, never from `{}`: several Polaris endpoints accept an empty object and answer 2xx.

In [10]:
PHASE_D = run_phase("D", [c for c in GRID if c.phase == "D"])
for target in (409, 400):
    rows = [r for r in PHASE_D if r["target"] == target]
    hit = sum(1 for r in rows if r["verdict"] == mx.COVERED)
    print(f"  {target}: {hit}/{len(rows)} covered")
    for r in rows:
        if r["verdict"] != mx.COVERED:
            print(f"    MISS  {r['op_id']:34} got {r['status']}")

  phase D: 43 cells; waiting 26s for a fresh window 
           0.3s in window 2026-09-10T02:37:30Z   {'covered': 22, 'missed': 21}
  409: 5/15 covered
    MISS  createNamespace                    got 403
    MISS  dropNamespace                      got 403
    MISS  registerTable                      got 403
    MISS  createTable                        got 403
    MISS  updateTable                        got 403
    MISS  createView                         got 403
    MISS  replaceView                        got 404
    MISS  renameTable                        got 404
    MISS  commitTransaction                  got 403
    MISS  renameView                         got 404
  400: 17/28 covered
    MISS  getToken                           got 500
    MISS  createNamespace                    got 500
    MISS  updateProperties                   got 403
    MISS  updateTable                        got 403
    MISS  planTableScan                      got 404
    MISS  fetchScanTasks        

## 7 -- Phase E: ONE commit, alone in its window

**Gate 2 is the feature. Everything else tests plumbing.**

`last_write_bytes` on the table row must equal the `response_size` of the **actual commit** that
produced it. "The last 2xx POST in that window" is only unambiguous when the window holds one,
which is why this phase is one call.

**FAILS IF** it matches the *first* commit (last-wins broken), or a 500's size (the 2xx filter
broken), or 0 (the `> 0` guard broken), or the field is absent when a commit did happen.

In [11]:
from iceberg_rest import IcebergREST as _IR

run_ic = _IR(ptu.POLARIS_URL, ptu.REALM, token=TOKENS[mx.RUNNER])

_wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
print(f"  waiting {_wait:.0f}s so the commit is alone in its window")
time.sleep(_wait)
_t0 = time.time()
_rid = mx.request_id(RUN, 3001, "gate2-the-commit")
lc.tag_clients([run_ic], _rid)
_resp = run_ic.commit_table(FX.cat, FX.ns, FX.tbl,
                            [{"action": "set-properties", "updates": {"mx.gate2": RUN}}])
lc.tag_clients([run_ic], None)
GATE2 = {
    "request_id": _rid, "status": _resp.status_code,
    "response_size": len(_resp.content or b""),
    "window": osr.window_bounds(_t0, WINDOW_SECONDS)[0],
    "window_end": osr.window_bounds(_t0, WINDOW_SECONDS)[1],
    "table": FX.tbl,
}
PHASE_WINDOWS["E"] = {"start": GATE2["window"], "end": GATE2["window"],
                      "straddled": False, "seconds": round(time.time() - _t0, 1), "cells": 1}
print(f"  commit {GATE2['status']}, {GATE2['response_size']} bytes, window {GATE2['window']}")
print("  NOTE the byte count above is what the CLIENT received. The gate compares the report's")
print("       claim against the ACCESS LOG's response_size, not against this -- they should")
print("       agree, and if they do not the access log is the authority.")

  waiting 8s so the commit is alone in its window
  commit 403, 256 bytes, window 2026-09-10T02:38:00Z
  NOTE the byte count above is what the CLIENT received. The gate compares the report's
       claim against the ACCESS LOG's response_size, not against this -- they should
       agree, and if they do not the access log is the authority.


## 8 -- Phase F: ONE delete, alone in its window

Gate 2's negative half, which is the one people skip. A window whose only write was a `DELETE`
(204, empty body) must carry **no** `last_write_bytes` at all -- **absent, not 0**. A query
looking for `: 0` would pass against a schema doing the thing the schema forbids, so the check
is `must_not exists`.

In [12]:
_drop_tbl = f"mx_{RUN}_droponly"
_r = run_ic.create_table(FX.cat, FX.ns, build_create_table_payload(_drop_tbl, SCHEMA))
print(f"  created {_drop_tbl}: {_r.status_code}  (in the PREVIOUS window, deliberately)")

_wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
print(f"  waiting {_wait:.0f}s so the DELETE is alone")
time.sleep(_wait)
_t0 = time.time()
_rid = mx.request_id(RUN, 3002, "gate2-the-delete")
lc.tag_clients([run_ic], _rid)
_resp = run_ic.drop_table(FX.cat, FX.ns, _drop_tbl)
lc.tag_clients([run_ic], None)
GATE2_NEG = {
    "request_id": _rid, "status": _resp.status_code,
    "response_size": len(_resp.content or b""), "table": _drop_tbl,
    "window": osr.window_bounds(_t0, WINDOW_SECONDS)[0],
}
PHASE_WINDOWS["F"] = {"start": GATE2_NEG["window"], "end": GATE2_NEG["window"],
                      "straddled": False, "seconds": round(time.time() - _t0, 1), "cells": 1}
print(f"  drop {GATE2_NEG['status']}, {GATE2_NEG['response_size']} bytes, "
      f"window {GATE2_NEG['window']}")
if GATE2_NEG["response_size"] != 0:
    print("  !! the DELETE returned a body. The 'size > 0' rule is what separates commit from")
    print("     drop, so a non-empty 204 makes this negative case VOID rather than failed.")

  created mx_1789007773_droponly: 403  (in the PREVIOUS window, deliberately)
  waiting 24s so the DELETE is alone
  drop 404, 118 bytes, window 2026-09-10T02:38:30Z
  !! the DELETE returned a body. The 'size > 0' rule is what separates commit from
     drop, so a non-empty 204 makes this negative case VOID rather than failed.


## 9 -- Phase G: grants, and a denied read on the same role

Gate 4. **`writes` on a `catalog-role` row is the number of privileges granted in that window** --
grants fold into the role they are granted on, because `classify` returns the matched span.

Then the denial case: a **403** on a catalog role with no successful request in the same window
must **still** produce that role's row, with `auth_denied: 1`, and increment `role_keys_forced`
on the summary. Role kinds are the one exception to the `create=false` guard; if the 403 lands
in `__errors__` instead, the `ROLE_KINDS` exemption is not working.

In [13]:
GRANTS = ["TABLE_READ_DATA", "TABLE_WRITE_DATA", "NAMESPACE_LIST"]
_crole = BINDING["catalogRoleName"]

_wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
print(f"  waiting {_wait:.0f}s")
time.sleep(_wait)
_t0 = time.time()

granted = []
for i, priv in enumerate(GRANTS):
    _rid = mx.request_id(RUN, 3100 + i, f"gate4-grant-{priv}")
    lc.tag_clients([adm_pc], _rid)
    r = adm_pc.grant_privilege(FX.cat, _crole, priv)
    lc.tag_clients([adm_pc], None)
    granted.append({"privilege": priv, "status": r.status_code, "request_id": _rid})
    print(f"    grant {priv:22} {r.status_code}")

# The denied principal reads the role it has no grants on. No successful request
# to this role from this identity in this window -- which is the whole point.
denied_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=TOKENS[mx.DENIED])
_rid = mx.request_id(RUN, 3199, "gate4-denied-role-read")
lc.tag_clients([denied_pc], _rid)
_dr = denied_pc.get_catalog_role(FX.cat, _crole)
lc.tag_clients([denied_pc], None)
print(f"    denied read of the role: {_dr.status_code}")

GATE4 = {
    "role": _crole, "granted": granted,
    "expected_writes": sum(1 for g in granted if 200 <= (g["status"] or 0) < 300),
    "denied_status": _dr.status_code, "denied_request_id": _rid,
    "window": osr.window_bounds(_t0, WINDOW_SECONDS)[0],
}
PHASE_WINDOWS["G"] = {"start": GATE4["window"], "end": osr.window_bounds(time.time(), WINDOW_SECONDS)[0],
                      "straddled": GATE4["window"] != osr.window_bounds(time.time(), WINDOW_SECONDS)[0],
                      "seconds": round(time.time() - _t0, 1), "cells": len(granted) + 1}
print(f"  expected writes on {_crole}: {GATE4['expected_writes']}  window {GATE4['window']}")

  waiting 29s
    grant TABLE_READ_DATA        201
    grant TABLE_WRITE_DATA       201
    grant NAMESPACE_LIST         201
    denied read of the role: 403
  expected writes on mx_1789007773_crole: 3  window 2026-09-10T02:39:30Z


## 10 -- Phase H: the 500 ladder

`lc.provokers_500`, in its own pure window so `errors_4xx == 0` is a **prediction** rather than
an observation. Every rung answered a **4xx** on 2026-09-07 -- a broken storage endpoint is 422
here, a missing bucket 400, a stale `entityVersion` 409 -- so **NOT PROVOKED is the expected
result and it is the contract working**, not a gap to paper over.

In [14]:
PROVOKERS = lc.provokers_500(
    adm_pc, run_ic, RUN, bucket=ptu.BUCKET, endpoint=ptu.MINIO_ENDPOINT,
    endpoint_internal=ptu.MINIO_ENDPOINT_INTERNAL,
    table_payload=lambda name: build_create_table_payload(name, SCHEMA), repeat=3)
print(f"  ladder: {[p.name for p in PROVOKERS]}")

_wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
print(f"  waiting {_wait:.0f}s so the burst is pure")
time.sleep(_wait)
_t0 = time.time()
_r500 = lc.drive_500([adm_pc, run_ic], RUN, 3200, PROVOKERS,
                     on_rung=lambda e: print(f"    {e['rung']:22} calls={e['calls']} "
                                             f"statuses={e['statuses']} provoked={e['provoked']}"))
_t1 = time.time()

FIVE_HUNDRED = {
    "winner": _r500["winner"], "rows": _r500["rows"], "ladder": _r500["ladder"],
    "driven_500": len(lc.classify_500s(_r500["rows"])["deliberate"]),
    "window": osr.window_bounds(_t0, WINDOW_SECONDS)[0],
    "straddled": osr.window_bounds(_t0, WINDOW_SECONDS)[0]
                 != osr.window_bounds(_t1, WINDOW_SECONDS)[0],
}
PHASE_WINDOWS["H"] = {"start": FIVE_HUNDRED["window"], "end": osr.window_bounds(_t1, WINDOW_SECONDS)[0],
                      "straddled": FIVE_HUNDRED["straddled"],
                      "seconds": round(_t1 - _t0, 1), "cells": len(_r500["rows"])}

if FIVE_HUNDRED["winner"]:
    print(f"  PROVOKED by `{FIVE_HUNDRED['winner']}`: {FIVE_HUNDRED['driven_500']} deliberate 500(s)")
else:
    print("  NOT PROVOKED -- no rung returned >= 500. Expected on this build: "
          "IcebergExceptionMapper\n     catches storage failures and maps them to 4xx. "
          "errors_5xx is NOT exercised this run,\n     and the accidental PG-HA 500s are not a "
          "substitute -- they are writes that COMMITTED.")
if FIVE_HUNDRED["straddled"]:
    print("  !! the burst straddled a boundary: the pure-window checks are VOID. Re-drive rather")
    print("     than reasoning about the split.")

  ladder: ['black_hole_endpoint', 'unresolvable_host', 'nonexistent_bucket', 'stale_entity_version']
  waiting 19s so the burst is pure
    black_hole_endpoint    calls=3 statuses=[404, 404, 404] provoked=0
    unresolvable_host      calls=3 statuses=[404, 404, 404] provoked=0
    nonexistent_bucket     calls=3 statuses=[404, 404, 404] provoked=0
    stale_entity_version   calls=3 statuses=[409, 409, 409] provoked=0
  NOT PROVOKED -- no rung returned >= 500. Expected on this build: IcebergExceptionMapper
     catches storage failures and maps them to 4xx. errors_5xx is NOT exercised this run,
     and the accidental PG-HA 500s are not a substitute -- they are writes that COMMITTED.


## 11 -- Phase I: cleanup

The cleanup DELETEs are **part of the test**, not tidying: `dropTable`, `dropView`,
`dropNamespace`, `deleteCatalog`, `deletePrincipal` are five of the 63 operations and rule 4
keeps every one of them.

In [15]:
CLEANUP = {}
for label, fn in [
    ("drop view", lambda: run_ic.drop_view(FX.cat, FX.ns, FX.view)),
    ("drop table", lambda: run_ic.drop_table(FX.cat, FX.ns, FX.tbl)),
    ("drop namespace", lambda: run_ic.drop_namespace(FX.cat, FX.ns)),
    ("delete catalog role", lambda: adm_pc.delete_catalog_role(FX.cat, BINDING["catalogRoleName"])),
    ("delete catalog", lambda: adm_pc.delete_catalog(FX.cat)),
]:
    try:
        CLEANUP[label] = fn().status_code
    except Exception as exc:
        CLEANUP[label] = f"{type(exc).__name__}: {exc}"
    print(f"  {label:22} {CLEANUP[label]}")

for name, role in ((RUNNER_NAME, RUNNER_ROLE), (DENIED_NAME, DENIED_ROLE)):
    try:
        lc.deprovision_run_principal(adm_pc, name, role)
        print(f"  deprovisioned {name}")
    except Exception as exc:
        print(f"  deprovision {name}: {type(exc).__name__}: {exc}")

# Anything the grid created that the fixture teardown does not know about.
for label, fn in [
    ("delete new principal", lambda: adm_pc.delete_principal(BINDING["new_principal"])),
    ("delete new principal role", lambda: adm_pc.delete_principal_role(BINDING["new_principal_role"])),
    ("delete new catalog", lambda: adm_pc.delete_catalog(BINDING["new_catalog"])),
]:
    try:
        print(f"  {label:26} {fn().status_code}")
    except Exception as exc:
        print(f"  {label:26} {type(exc).__name__}")

  drop view              404
  drop table             403
  drop namespace         403
  delete catalog role    204
  delete catalog         400
  deprovisioned mx_1789007773_runner
  deprovisioned mx_1789007773_denied
  delete new principal       204
  delete new principal role  204
  delete new catalog         204


## 12 -- Collection

Every call carried a `Polaris-Request-Id`, so the per-call half is a join rather than a guess.
`settle` waits for the count to stop moving: ingest is asynchronous and a count taken too early
is a statement about the clock.

**Scoped to the run's own windows.** An unscoped count over a continuously written index differs
from itself by elapsed time -- one row per window, two a minute -- which on 2026-09-10 nearly
became a finding about 55 rows that were never missing.

In [16]:
WINDOWS = sorted({w["start"] for w in PHASE_WINDOWS.values()}
                 | {w["end"] for w in PHASE_WINDOWS.values()})
RUN_WINDOW = (WINDOWS[0], WINDOWS[-1])
print(f"  the run touched windows {RUN_WINDOW[0]} .. {RUN_WINDOW[1]}  ({len(WINDOWS)} distinct)")
for name, w in sorted(PHASE_WINDOWS.items()):
    print(f"    phase {name}  {w['start']}  {w['seconds']:>6.1f}s  {w['cells']:>3} cells"
          + ("   STRADDLED" if w["straddled"] else ""))

_body = {"size": 0, "query": {"bool": {"filter": [
    {"prefix": {osr.kw("mdc.requestId"): f"nb-{RUN}-"}}]}}}
settled = OS.settle(_body, index=OS.logs_index, quiet_for=6, timeout=120, interval=3)
print(f"\n  records carrying this run's ids: {settled.total}")

FOUND = {}
for row in ROWS:
    rid = row["request_id"]
    if rid:
        FOUND[rid] = OS.for_request_id(rid)

_with_id = [r for r in ROWS if r["request_id"]]
_recovered = [r for r in _with_id if FOUND.get(r["request_id"])]
print(f"  correlation: {len(_recovered)}/{len(_with_id)} calls recovered by request id")

# THE SPLIT THAT MATTERS. A KEPT call must be findable and a miss there is a real
# fault; a COUNTED call is findable only incidentally, through whatever application
# lines it happened to emit. The unsplit ratio punishes the policy for working.
KEPT_TARGETS = {401, 403, 404, 409, 400, 500}
_kept = [r for r in _with_id if r["target"] in KEPT_TARGETS or (r["status"] or 0) >= 400]
_kept_found = [r for r in _kept if FOUND.get(r["request_id"])]
print(f"  of those, errors (rule 3 keeps ALL of them -- an ASSERTION): "
      f"{len(_kept_found)}/{len(_kept)}")
MISSING_ERRORS = [r["label"] for r in _kept if not FOUND.get(r["request_id"])]
if MISSING_ERRORS:
    print(f"  !! NOT FOUND, and rule 3 says they must be: {MISSING_ERRORS[:8]}")

  the run touched windows 2026-09-10T02:36:30Z .. 2026-09-10T02:40:00Z  (7 distinct)
    phase B  2026-09-10T02:36:30Z     1.0s   63 cells
    phase C  2026-09-10T02:37:00Z     1.2s  180 cells
    phase D  2026-09-10T02:37:30Z     0.3s   43 cells
    phase E  2026-09-10T02:38:00Z     0.0s    1 cells
    phase F  2026-09-10T02:38:30Z     0.0s    1 cells
    phase G  2026-09-10T02:39:30Z     0.1s    4 cells
    phase H  2026-09-10T02:40:00Z     0.5s   12 cells

  records carrying this run's ids: 2661
  correlation: 286/286 calls recovered by request id
  of those, errors (rule 3 keeps ALL of them -- an ASSERTION): 254/254


## 13 -- The matrix

One row per cell. `target` is what the cell asked for, `status` is what it got, and a row where
they differ is **MISSED with the actual code** -- never recounted as coverage of whatever it hit.

In [17]:
import pandas as pd

M = pd.DataFrame(ROWS)
RESULTS = {(r["op_id"], r["target"]): {"verdict": r["verdict"], "actual": r["status"],
                                       "why": r["why"]} for r in ROWS}
COVERAGE = mx.coverage_by_operation(OPS, RESULTS)

print("  by target:")
for target in sorted({r["target"] for r in ROWS}, key=str):
    rows = [r for r in ROWS if r["target"] == target]
    cov = sum(1 for r in rows if r["verdict"] == mx.COVERED)
    err = sum(1 for r in rows if r["verdict"] == mx.ERROR)
    print(f"    {str(target):>4}  {cov:>3}/{len(rows):<3} covered"
          + (f"   {err} transport error(s)" if err else ""))

_full = [c for c in COVERAGE if not c["missed"] and not c["not_driven"]]
print(f"\n  operations with EVERY planned cell covered: {len(_full)}/{len(OPS)}")

_holes = [c for c in COVERAGE if c["missed"]]
print(f"  operations with a missed cell: {len(_holes)}")
for c in _holes[:25]:
    got = {t: RESULTS[(c['op_id'], t)]['actual'] for t in c['missed']}
    print(f"    {c['op_id']:34} missed {c['missed']}  got {got}")

pd.set_option("display.width", 200, "display.max_rows", 400, "display.max_colwidth", 44)
print("\n" + M[["op_id", "method", "target", "status", "verdict", "principal", "phase",
                "elapsed_ms"]].to_string(index=False))

  by target:
       2   32/63  covered
     400   17/28  covered
     401   59/63  covered
     403   45/62  covered
     404   54/55  covered
     409    5/15  covered

  operations with EVERY planned cell covered: 23/63
  operations with a missed cell: 40
    getConfig                          missed [403]  got {403: 200}
    getToken                           missed [400]  got {400: 500}
    listNamespaces                     missed [2]  got {2: 403}
    createNamespace                    missed [2, 400, 409]  got {2: 403, 400: 500, 409: 403}
    dropNamespace                      missed [2, 409]  got {2: 403, 409: 403}
    loadNamespaceMetadata              missed [2]  got {2: 403}
    namespaceExists                    missed [2]  got {2: 403}
    updateProperties                   missed [2, 400]  got {2: 403, 400: 403}
    registerTable                      missed [2, 409]  got {2: 403, 409: 403}
    listTables                         missed [2]  got {2: 403}
    createTable    

## 14 -- Gates 1 to 6

Verbatim from `GUIDE-schema-v3-testing.md`, through `os_report` so the schema filter, the
`value_count` pairing and the `.keyword` suffixes cannot be forgotten at a call site.

Each prints PASS / FAIL / **VOID**. VOID is not a soft FAIL: it means the window carried nothing
that could answer the question, and calling that a pass invents a measurement while calling it a
failure invents a fault.

In [18]:
GATES = {}


def verdict(name, ok, detail=""):
    GATES[name] = {"ok": ok, "detail": detail}
    mark = {True: "PASS", False: "FAIL", None: "VOID"}[ok]
    print(f"  {name:34} {mark}   {detail}")


# -- Gate 1: the margin invariants ---------------------------------------
MARGINS = OS.margins(RUN_WINDOW)
_answered = [w for w in MARGINS if w["ok"] is not None]
_bad = [w for w in _answered if not w["ok"]]
verdict("Gate 1 margins",
        None if not _answered else not _bad,
        f"{len(_answered)}/{len(MARGINS)} windows answered"
        + ("" if not _bad else "; " + "; ".join(f"{w['window_start']}: {w['why']}" for w in _bad[:3])))

# -- Gate 0b: the array split, on THIS side ------------------------------
_split = OS.report_types(window=RUN_WINDOW)
# VOID, not FAIL, when the window held no report rows at all: a window with
# access_seen == 0 has no resource or principal rows to split INTO, and calling
# that a failure invents a pipeline fault.
_split_ok = None if not _split else sorted(_split) == ["principal", "resource", "summary"]
verdict("Gate 0b array split", _split_ok, str(_split))

# -- Gate 2: last_write_bytes, THE FEATURE -------------------------------
_claim = OS.report(osr.q_last_write(resource=None, kind="table",
                                    window=(GATE2["window"], GATE2["window"]))).sources
_records = OS.logs(osr.q_access_records(GATE2["window"], GATE2["window_end"],
                                        method="POST", path_wildcard="*/tables/*")).sources
_2xx = [r for r in _records if 200 <= int(r.get("http_status", 0)) < 300]
if not _claim:
    verdict("Gate 2 last_write_bytes", None,
            "no table row carried last_write_bytes in the commit's window -- "
            + ("the commit itself failed" if GATE2["status"] >= 400 else "UNPROVEN"))
elif not _2xx:
    verdict("Gate 2 last_write_bytes", None,
            "the report claims a value and the access log has no 2xx POST to compare it to")
else:
    _last = sorted(_2xx, key=lambda r: r.get("@timestamp"))[-1]
    _want = int(_last.get("response_size") or 0)
    _got = int(_claim[0].get("last_write_bytes") or 0)
    verdict("Gate 2 last_write_bytes", _got == _want,
            f"report={_got} last 2xx POST response_size={_want} on {_claim[0].get('resource')}")

# -- Gate 2, the negative half -------------------------------------------
_absent = OS.report(osr.q_absent_last_write(GATE2_NEG["table"],
                                            (GATE2_NEG["window"], GATE2_NEG["window"])))
_present = OS.report(osr.q_last_write(resource=GATE2_NEG["table"], kind=None,
                                      window=(GATE2_NEG["window"], GATE2_NEG["window"])))
verdict("Gate 2 delete leaves it ABSENT",
        None if not (_absent.total or _present.total) else _present.total == 0,
        f"rows without the field={_absent.total}  rows WITH it={_present.total} (want 0)")

# -- Gate 3: classification ----------------------------------------------
_cls = OS.report(osr.q_classification(window=RUN_WINDOW)).aggs
_pairs = {b["key"]: [k["key"] for k in b["kind"]["buckets"]]
          for b in (_cls.get("api") or {}).get("buckets", [])}
_kinds = {k for v in _pairs.values() for k in v}
verdict("Gate 3 classification",
        None if not _pairs else ("management" not in _kinds),
        f"{_pairs}" + ("   resource_kind must never read 'management'"
                       if "management" in _kinds else ""))

# -- Gate 4: roles and the privilege count -------------------------------
_roles = OS.report(osr.q_role_rows(kind="catalog-role",
                                   window=(GATE4["window"], GATE4["window"]))).sources
_mine = [r for r in _roles if GATE4["role"] in str(r.get("resource", ""))]
if not _mine:
    verdict("Gate 4 privilege count", None,
            f"no catalog-role row for {GATE4['role']} in {GATE4['window']}")
else:
    _w = int(_mine[0].get("writes") or 0)
    verdict("Gate 4 privilege count", _w == GATE4["expected_writes"],
            f"writes={_w} granted={GATE4['expected_writes']} on {_mine[0].get('resource')}")
    _denied = int(_mine[0].get("auth_denied") or 0)
    verdict("Gate 4 denial forces the row", _denied >= 1,
            f"auth_denied={_denied} (the ROLE_KINDS exemption; 0 means the 403 fell to __errors__)")

_grants_row = [r for r in _roles if str(r.get("resource", "")).endswith("/grants")]
verdict("Gate 4 grants fold into the role", not _grants_row,
        "a .../grants row of its own means the fold broke" if _grants_row else "no separate grants row")

# -- Gate 5: the two synthetic buckets -----------------------------------
_syn = OS.synthetic_split(window=RUN_WINDOW)
verdict("Gate 5 __errors__ holds only errors", _syn["verdict"], _syn["why"] or
        f"requests={_syn['errors']['requests']!r}")
_other_req = _syn["other"]["requests"]
verdict("Gate 5 __other__ is overflow only",
        None if _other_req.absent else _other_req.as_int() == 0,
        f"__other__ requests={_other_req!r} -- non-zero means a window really carried 500+ "
        "distinct resources, or the split regressed")

# -- Gate 6: what still needs a classification rule ----------------------
UNCLASSIFIED = OS.unclassified(window=RUN_WINDOW)
verdict("Gate 6 every path has a rule", not UNCLASSIFIED,
        "only the synthetic keys came back" if not UNCLASSIFIED
        else f"{len(UNCLASSIFIED)} real path(s) with no RESOURCE_PATTERNS entry")
for u in UNCLASSIFIED[:20]:
    print(f"      {u['api_kind']:12} {u['resource']}  ({u['rows']} row(s))")

  Gate 1 margins                     VOID   0/0 windows answered
  Gate 0b array split                VOID   {}
  Gate 2 last_write_bytes            VOID   no table row carried last_write_bytes in the commit's window -- the commit itself failed
  Gate 2 delete leaves it ABSENT     VOID   rows without the field=0  rows WITH it=0 (want 0)
  Gate 3 classification              VOID   {}
  Gate 4 privilege count             VOID   no catalog-role row for mx_1789007773_crole in 2026-09-10T02:39:30Z
  Gate 4 grants fold into the role   PASS   no separate grants row
  Gate 5 __errors__ holds only errors VOID   __errors__ has no rows in this scope -- unproven
  Gate 5 __other__ is overflow only  VOID   __other__ requests=<Metric requests ABSENT (0 docs carried it)> -- non-zero means a window really carried 500+ distinct resources, or the split regressed
  Gate 6 every path has a rule       PASS   only the synthetic keys came back


## 15 -- The schema's own invariants

From `SCHEMA-report.md`, *Invariants worth asserting*. The fourth one --
`max_record_time - min_record_time <= window_seconds` -- is checked **client-side**: the fields
are RFC3339 strings and parse fine here, whatever OpenSearch typed them as. What the `text`
mapping kills is date maths *inside* the index, not the invariant.

In [19]:
INVARIANTS = []


def inv(name, ok, detail=""):
    INVARIANTS.append({"name": name, "ok": ok, "detail": detail})
    mark = {True: "PASS", False: "FAIL", None: "VOID"}[ok]
    print(f"  {name:44} {mark}   {detail}")


for w in MARGINS:
    if w["ok"] is None:
        continue
    inv(f"margin {w['window_start']}", w["ok"], w["why"] or
        f"{w['resource_requests'].as_int(0)} == {w['expected']}")

# carried_rows == the number of rows with requests == 0
_zero = OS.report({"size": 0, "query": {"bool": {"filter":
        osr._filters({"range": {"requests": {"lte": 0}}},
                     schema_version=OS.schema_version, window=RUN_WINDOW)}}})
_carried = [w["carried_rows"] for w in MARGINS if not w["carried_rows"].absent]
inv("carried_rows counts the zero-request rows",
    None if not _carried else True,
    f"rows with requests<=0: {_zero.total}; summary carried_rows total: "
    f"{sum(m.as_int(0) for m in _carried)}  -- v3 merges resources AND principals into one "
    "counter (review #3), so this is reported, not asserted")

# report_seq: contiguous per (hostname, schema_version)
_seqs = OS.report(osr.q_reports(report_type="summary", window=RUN_WINDOW, size=500,
                                source=["report_seq", "hostname", "window_start"])).sources
_by_host = {}
for r in _seqs:
    _by_host.setdefault(r.get("hostname"), []).append(int(r.get("report_seq") or 0))
for host, seqs in _by_host.items():
    seqs = sorted(seqs)
    contiguous = seqs and (max(seqs) - min(seqs) + 1 == len(set(seqs)))
    inv(f"report_seq contiguous on {host}", bool(contiguous),
        f"{min(seqs)}..{max(seqs)} over {len(set(seqs))} row(s)"
        + ("" if contiguous else "   A GAP: a window closed with no tick, or the pod restarted"))

# min/max record time, parsed client-side
import datetime as _dt

_rt = OS.report(osr.q_reports(report_type="summary", window=RUN_WINDOW, size=200,
                              source=["min_record_time", "max_record_time", "window_start",
                                      "window_seconds"])).sources
_checked, _bad = 0, []
for r in _rt:
    lo, hi = r.get("min_record_time"), r.get("max_record_time")
    if not lo or not hi:
        continue
    try:
        span = (_dt.datetime.fromisoformat(hi.replace("Z", "+00:00"))
                - _dt.datetime.fromisoformat(lo.replace("Z", "+00:00"))).total_seconds()
    except ValueError:
        continue
    _checked += 1
    if span > int(r.get("window_seconds") or WINDOW_SECONDS):
        _bad.append((r.get("window_start"), span))
inv("record span fits inside the window", None if not _checked else not _bad,
    f"{_checked} window(s) carried both timestamps"
    + (f"; OVER: {_bad[:3]}" if _bad else "")
    + ("   (idle windows write \"\" and are skipped -- that is review #2, not a pass)"
       if _checked < len(_rt) else ""))

  carried_rows counts the zero-request rows    VOID   rows with requests<=0: 0; summary carried_rows total: 0  -- v3 merges resources AND principals into one counter (review #3), so this is reported, not asserted
  record span fits inside the window           VOID   0 window(s) carried both timestamps


## 16 -- Findings, and the report for `local-k8s`

Two documents, written by this cell.

`doc-api-status-matrix-results.md` is the run of record for **this** repo. `REPORT-for-local-k8s.md`
is the work list for the **pipeline** repo: only the items whose fix lives there -- a
classification rule, an index template, a gate that cannot fail -- each with what was measured
and what to change.

Test-driven, as agreed: the run does not fix the pipeline, it produces the list.

In [20]:
FINDINGS = []


def finding(where, title, measured, remedy, severity="finding"):
    FINDINGS.append({"where": where, "title": title, "measured": measured,
                     "remedy": remedy, "severity": severity})


# 1. Endpoints with no classification rule -- the reason the whole grid exists.
if UNCLASSIFIED:
    finding("local-k8s", "endpoints with no RESOURCE_PATTERNS rule",
            "; ".join(f"{u['api_kind']} {u['resource']}" for u in UNCLASSIFIED[:12]),
            "add a RESOURCE_PATTERNS entry per path, then re-run Gate 6. These are visible "
            "ONLY because the grid drove all 63 operations -- Gate 6 has never had traffic on "
            "them before.")

# 2. The mapping traps.
for note in MAPPING_NOTES:
    finding("local-k8s", "index mapping", note,
            "an index template for polaris-report-* typing min_record_time / max_record_time as "
            "date, and the numeric report fields as long. A template survives the Lua writing "
            "\"\" on idle windows; waiting for a fresh index does not, because the first window "
            "of a day is almost always idle at 30s windows.")

# 3. Gates that cannot fail as written.
finding("guide", "Gate 5's term filter matches nothing as written",
        'GUIDE-schema-v3-testing.md Gate 5 uses {"term": {"resource": "__errors__"}}. On a '
        "dynamic-mapped index that runs against the analysed text field, where the standard "
        "analyser has already turned __errors__ into the token `errors`.",
        "use resource.keyword (Gate 2's resource_kind filter has the same shape). Until then "
        "that gate reports zero and passes without having looked.")

# 4. Whatever the gates actually found.
for name, g in GATES.items():
    if g["ok"] is False:
        finding("local-k8s", f"{name} FAILED", g["detail"],
                "see the gate's stated failure mode in GUIDE-schema-v3-testing.md")
    elif g["ok"] is None:
        finding("this repo", f"{name} VOID", g["detail"],
                "the window carried nothing that could answer it -- re-drive, do not re-read")

# 5. Cells that missed, grouped by what they hit instead.
_miss_by = {}
for r in ROWS:
    if r["verdict"] == mx.MISSED:
        _miss_by.setdefault((r["target"], r["status"]), []).append(r["op_id"])
for (target, actual), ops_ in sorted(_miss_by.items(), key=str):
    finding("this repo", f"{len(ops_)} cell(s) targeting {target} returned {actual}",
            ", ".join(ops_[:10]), "either the driver is wrong or this build does not produce "
            f"{target} on those endpoints -- decide which, and record it in the ledger")

# 6. Errors that rule 3 promises to keep and did not.
if MISSING_ERRORS:
    finding("local-k8s", "rule 3 kept fewer errors than it promises",
            f"{len(MISSING_ERRORS)} error call(s) left no record: {MISSING_ERRORS[:8]}",
            "rule 3 keeps every http_status >= 400 with no cap. A missing one is a real gap "
            "in the retention policy or in the shipper's delivery.", severity="SERIOUS")

print(f"  {len(FINDINGS)} finding(s)")
for f in FINDINGS:
    print(f"    [{f['where']:>10}] {f['title']}")

STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%MZ")


def _table(rows, cols):
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for r in rows:
        out.append("| " + " | ".join(str(r.get(c, "")) for c in cols) + " |")
    return "\n".join(out)


results = [
    f"# API status matrix -- run {RUN}", "",
    f"Driven {STAMP} against `local`, schema v{OS.schema_version}, "
    f"windows of {WINDOW_SECONDS}s.", "",
    f"- **{len(OPS)} operations**, **{len(GRID)} cells**, {len(ROWS)} calls issued",
    f"- covered: {sum(1 for r in ROWS if r['verdict'] == mx.COVERED)}   "
    f"missed: {sum(1 for r in ROWS if r['verdict'] == mx.MISSED)}   "
    f"transport errors: {sum(1 for r in ROWS if r['verdict'] == mx.ERROR)}",
    f"- correlation on error cells (rule 3, an assertion): "
    f"{len(_kept_found)}/{len(_kept)}",
    f"- 500 ladder: {FIVE_HUNDRED['winner'] or 'NOT PROVOKED (expected on this build)'}",
    "", "## Gates", "",
    _table([{"gate": k, "verdict": {True: "PASS", False: "FAIL", None: "VOID"}[v["ok"]],
             "detail": v["detail"]} for k, v in GATES.items()], ["gate", "verdict", "detail"]),
    "", "## Windows", "",
    _table([{"phase": k, **v} for k, v in sorted(PHASE_WINDOWS.items())],
           ["phase", "start", "seconds", "cells", "straddled"]),
    "", "## Coverage by operation", "",
    _table(COVERAGE, ["op_id", "method", "api", "covered", "missed", "not_driven", "declared"]),
    "", "## Not reachable", "",
    _table(mx.ledger(), ["status", "state", "why"]),
]
pathlib.Path("doc-api-status-matrix-results.md").write_text("\n".join(results), encoding="utf-8")

report = [
    "# For `local-k8s`: what this run says needs changing there", "",
    f"Produced by `log-coverage/polaris_log_coverage_v2.ipynb`, run {RUN}, {STAMP}.",
    "Only items whose fix lives in the pipeline repo are here; everything else stays in",
    "`doc-api-status-matrix-results.md`.", "",
    f"**How it was driven:** {len(OPS)} operations from the vendored OpenAPI documents, "
    f"{len(GRID)} (operation x status) cells, boundary-aligned phases so the per-window gates "
    "mean something.", "",
]
for i, f in enumerate([f for f in FINDINGS if f["where"] == "local-k8s"], 1):
    report += [f"## {i}. {f['title']}" + ("  **SERIOUS**" if f["severity"] == "SERIOUS" else ""),
               "", f"**Measured:** {f['measured']}", "", f"**Change:** {f['remedy']}", ""]
_guide = [f for f in FINDINGS if f["where"] == "guide"]
if _guide:
    report += ["## For `GUIDE-schema-v3-testing.md` itself", ""]
    for f in _guide:
        report += [f"### {f['title']}", "", f"**Measured:** {f['measured']}", "",
                   f"**Change:** {f['remedy']}", ""]
pathlib.Path("REPORT-for-local-k8s.md").write_text("\n".join(report), encoding="utf-8")

print("\n  wrote doc-api-status-matrix-results.md")
print("  wrote REPORT-for-local-k8s.md")

  25 finding(s)
    [ local-k8s] index mapping
    [ local-k8s] index mapping
    [     guide] Gate 5's term filter matches nothing as written
    [ this repo] Gate 1 margins VOID
    [ this repo] Gate 0b array split VOID
    [ this repo] Gate 2 last_write_bytes VOID
    [ this repo] Gate 2 delete leaves it ABSENT VOID
    [ this repo] Gate 3 classification VOID
    [ this repo] Gate 4 privilege count VOID
    [ this repo] Gate 5 __errors__ holds only errors VOID
    [ this repo] Gate 5 __other__ is overflow only VOID
    [ this repo] 1 cell(s) targeting 2 returned 400
    [ this repo] 19 cell(s) targeting 2 returned 403
    [ this repo] 11 cell(s) targeting 2 returned 404
    [ this repo] 1 cell(s) targeting 400 returned 200
    [ this repo] 2 cell(s) targeting 400 returned 403
    [ this repo] 3 cell(s) targeting 400 returned 404
    [ this repo] 5 cell(s) targeting 400 returned 500
    [ this repo] 4 cell(s) targeting 401 returned 404
    [ this repo] 1 cell(s) targeting 403 returne

## 17 -- Teardown

Close the sessions. Nothing here holds a Pgpool-II connection, but the HTTP sessions and the
frame are dropped explicitly so a re-run starts clean.

In [21]:
import gc

for _c in (adm_pc, adm_ic, run_ic, denied_pc):
    try:
        _c.session.close()
    except Exception:
        pass
try:
    OS.session.close()
except Exception:
    pass

del M
gc.collect()
print("  sessions closed, frame released")
print(f"\n  run {RUN} complete -- {len(ROWS)} calls, {len(GATES)} gates, {len(FINDINGS)} findings")

  sessions closed, frame released

  run 1789007773 complete -- 286 calls, 10 gates, 25 findings
